In [1]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [8]:
import pickle

def load_models():
    # Correct Syntax: variable = pickle.load(file)
    with open("../models/tfidf.pkl", "rb") as f:
        tfidf_obj = pickle.load(f)
    
    with open("../models/best_model.pkl", "rb") as f:
        model_obj = pickle.load(f)
        
    with open("../models/label_encoder.pkl", "rb") as f:
        encoder_obj = pickle.load(f)
        
    return tfidf_obj, model_obj, encoder_obj

# Now call it
tfidf, clf_model, label_encoder = load_models()

In [9]:
os.environ["GROQ_API_KEY"] = "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"

In [10]:
class AgentState(TypedDict):
    question: str
    category: str
    answer: str

In [11]:
def classification_node(state: AgentState):
    query = state['question']
    
    X = tfidf.transform([query])
    
    pred_numeric = clf_model.predict(X)
    
    category_name = label_encoder.inverse_transform(pred_numeric)[0]
    
    return {"category": category_name}

In [12]:
def answer_node(state: AgentState):
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
    
    prompt = ChatPromptTemplate.from_template(
        "You are an expert academic tutor. The student's question is about {category}.\n"
        "Question: {question}\n"
        "Provide a clear and helpful explanation:"
    )
    

    chain = prompt | llm
    response = chain.invoke({"category": state['category'], "question": state['question']})
    
    return {"answer": response.content}

In [13]:
workflow = StateGraph(AgentState)

workflow.add_node("classify", classification_node)
workflow.add_node("answer", answer_node)

workflow.set_entry_point("classify")
workflow.add_edge("classify", "answer")
workflow.add_edge("answer", END)

app = workflow.compile()

In [14]:
import sys
print(sys.executable)

c:\SmartStudyAssistant\.venv\Scripts\python.exe
